# Recover the five-seed verification after a rejected joint-weight solver fit

**Keep the failed run's full Kaggle output. Do not delete or overwrite it.** This notebook reuses the original worker, configuration, result context, backbone checkpoints, head checkpoints and successful frozen fits. It retries only missing work. The compact `verification_results.zip` excludes large backbone checkpoints and cannot restore training by itself.

**Same live session:** upload/paste these cells into that session and run them while `/kaggle/working/dpe_five_seed_verification` still exists. Do not rerun the original notebook's Run All.

**Finished/closed batch session:** save/retain the failed version's output, create a new recovery notebook with **T4 GPU + Internet**, and attach that version's **full output** as Kaggle Input. Upload this notebook and Run All. Auto-discovery finds a single prior full working directory; set `RESUME_FROM` below only if multiple candidates are attached. Attaching a full output as Input does not retrain it.

The traceback is in an additional-partition global `joint-l0.01` weight fit, not the CNN trainer. SciPy's convergence message did not satisfy the implementation's complete feasibility/optimality certificate. The log does not show which numerical check failed. Recovery leaves the scientific objective, penalty, probability floor, simplex constraint and **1e-7 acceptance tolerance** unchanged. It retries a rejected solve with tighter SLSQP precision and independently optimizes the certificate's four training-group multipliers. A certified global objective-gap bound is required; a success flag alone is insufficient.

Successful cached fits remain byte-for-byte intact. Recovery provenance, rejected-fit training inputs and every fallback certificate are written under `logs/` and included in the result ZIP. Completed backbones are reused; unfinished backbone training can resume from the last completed epoch. Missing backbones still require training. This notebook does not claim that both new seeds had finished before the error.

Validation:21 small solver cases; restore/live-session reuse; worker/config preservation; identity and runtime mismatch rejection; saved-partition preflight; subprocess success/failure handling; and reporting helpers passed lightweight checks. GPU/model calls were stubbed in orchestration tests. The actual failed partition tensors must still pass the Kaggle preflight; no full Kaggle execution or heavy experiment ran here.


In [ ]:
RESUME_FROM = None  # Optional: previous FULL dpe_five_seed_verification directory in /kaggle/input.
WORK_DIR = "/kaggle/working/dpe_five_seed_verification"  # Keep the ORIGINAL path.


In [ ]:
import hashlib,json,os,shutil,stat,subprocess,sys
from pathlib import Path
work=Path(WORK_DIR).resolve()
required=["config.json","five_seed_verification_runner.py"]
def is_full(p):return all((p/name).is_file() for name in required) and (p/"additional_sources").is_dir()
if not is_full(work):
    if work.exists() and any(work.iterdir()):
        raise RuntimeError("Nonempty working directory is incomplete. Preserve it; do not overwrite or delete it.")
    if RESUME_FROM:
        prior=Path(RESUME_FROM).resolve()
        if not is_full(prior):raise RuntimeError("RESUME_FROM must be a full prior working directory, not the compact ZIP.")
    else:
        candidates=sorted({p.parent for p in Path("/kaggle/input").rglob("five_seed_verification_runner.py") if is_full(p.parent)})
        if len(candidates)!=1:raise RuntimeError("Attach the FAILED run's full output. Set RESUME_FROM if multiple full outputs are attached. Found: "+str(candidates))
        prior=candidates[0]
    print("Restoring full output without changing the source:",prior)
    stage=work.with_name(work.name+".restore-in-progress")
    if stage.exists():
        for item in [stage,*stage.rglob("*")]:
            if not item.is_symlink():item.chmod(item.stat().st_mode|stat.S_IWUSR|(stat.S_IXUSR if item.is_dir() else 0))
    needed=sum(p.stat().st_size for p in prior.rglob("*") if p.is_file() and not (stage/p.relative_to(prior)).is_file())
    if shutil.disk_usage(work.parent).free < needed+512*1024**2:
        raise RuntimeError("Not enough free disk to restore full output safely. Required additional bytes: "+str(needed))
    shutil.copytree(prior,stage,dirs_exist_ok=True)
    # Kaggle Input may be read-only. Make only the RESTORED destination writable.
    for item in [stage,*stage.rglob("*")]:
        if not item.is_symlink():item.chmod(item.stat().st_mode|stat.S_IWUSR|(stat.S_IXUSR if item.is_dir() else 0))
    if work.exists():work.rmdir()  # Empty directory only; never remove saved artifacts.
    stage.rename(work)
else:
    print("Using existing working files; no restore required:",work)
config_path=work/"config.json";runner_path=work/"five_seed_verification_runner.py"
C=json.loads(config_path.read_text())
if C["WORK_DIR"]!=str(work):raise RuntimeError("Restore to the original WORK_DIR path; configuration must remain unchanged.")
if C.get("ORIGINAL_SEEDS")!=[0,1,2] or C.get("ADDITIONAL_SEEDS")!=[3,4]:raise RuntimeError("Unexpected original experiment configuration.")
EXPECTED_WORKER_HASH = "d0274451f1c424f52ecd46a7f0b3f61bbd89f23c5a9aec5cde6fb4179c4fb2dc"
if hashlib.sha256(runner_path.read_bytes()).hexdigest()!=EXPECTED_WORKER_HASH:
    raise RuntimeError("Worker differs from the reviewed five-seed notebook. Preserve all files and share its source for review.")
repo=work/"reference_repo";runtime=work/"runtime_packages"
if not runtime.is_dir():raise RuntimeError("Pinned runtime_packages is missing. Preserve the full output rather than using the compact ZIP.")
commit="8e14249f4b3510363be06f627182e2daa9926179"
if not (repo/".git").is_dir():
    subprocess.run(["git","init","-q",str(repo)],check=True)
    subprocess.run(["git","-C",str(repo),"remote","add","origin","https://github.com/chautanphat/waterbirds-shortcut-mitigation.git"],check=True)
    subprocess.run(["git","-C",str(repo),"fetch","--depth","1","origin",commit],check=True)
    # Record the original tree without overwriting restored source files.
    subprocess.run(["git","-C",str(repo),"reset","--mixed","FETCH_HEAD"],check=True,stdout=subprocess.DEVNULL)
head=subprocess.check_output(["git","-C",str(repo),"rev-parse","HEAD"],text=True).strip()
if head!=commit:raise RuntimeError("Restored reference checkout is not the original pinned commit.")
for path in sorted((work/"additional_sources/backbones/standard").glob("seed*")):
    print(path.name,"COMPLETE backbone" if (path/"summary.json").is_file() else "partial backbone; epoch checkpoint will be inspected by original trainer")
for path in sorted((work/"results").glob("*/status.json")):
    print("Prior status:",path,json.loads(path.read_text()))
print("Original worker/config unchanged. Completed source and frozen-fit identities will be checked before reuse.")


## Install the recovery adapter (no new packages)

The reference repository files are not edited. The adapter first calls the original solver. Only its explicit `Joint solver failed; no candidate accepted` rejection triggers the fallback. Data/shape/label errors remain errors. All fitting uses combiner-training predictions/labels; test performance is never used to accept a solver result. The worker itself remains unchanged, so previously successful cache identities can be checked and reused. The separate recovery provenance records this numerical adapter transparently.


In [ ]:
RECOVERY_SOURCE = r'''"""Kaggle recovery adapter. Keep the original worker/config/source files intact."""
import hashlib,json,os,runpy,sys,time
from pathlib import Path
import numpy as np
from scipy.optimize import minimize


def simplex_projection(v):
    ordered=np.sort(v)[::-1];theta=(np.cumsum(ordered)-1)/np.arange(1,len(v)+1)
    rho=np.flatnonzero(ordered>theta)[-1]
    return np.maximum(v-theta[rho],0)


def certified_refit(p,y,groups,penalty,solver,initial_weights=()):
    """Same convex minimax log-loss+L2 fit; independently check a global bound."""
    if not penalty>0:raise ValueError('Recovery is limited to the declared positive penalties')
    h,n,_=p.shape;u=np.full(h,1/h);tol=solver['feasibility_tolerance']
    correct=np.maximum(p[:,np.arange(n),np.asarray(y,dtype=int)],solver['probability_floor'])
    grouped=[correct[:,np.asarray(groups)==g] for g in ['00','01','10','11']]
    if any(q.shape[1]==0 for q in grouped):raise ValueError('Missing training group')
    def losses(w):return np.array([-np.log(w@q).mean() for q in grouped])
    def gradients(w):return np.array([-(q/(w@q)).mean(1) for q in grouped])
    def value(w):return float(losses(w).max()+penalty*np.square(w-u).sum())
    def certificate(w):
        loss=losses(w);G=gradients(w)
        def gap(alpha,with_gradient=False):
            grad=alpha@G+2*penalty*(w-u)
            # Minimize the tangent lower bound + exact quadratic on the simplex.
            v=simplex_projection(w-grad/(2*penalty))
            bound=float(loss.max()-alpha@loss+grad@(w-v)-penalty*np.square(v-w).sum())
            jac=-loss+G@(w-v)
            return (bound,jac) if with_gradient else bound
        alpha=np.zeros(4);alpha[np.argmax(loss)]=1
        # Optimize multipliers independently instead of trusting solver.v.
        dual=minimize(lambda a:gap(a,True),alpha,jac=True,method='SLSQP',bounds=[(0.,1.)]*4,
            constraints=[dict(type='eq',fun=lambda a:a.sum()-1,jac=lambda a:np.ones(4))],
            options=dict(ftol=1e-14,maxiter=2000))
        a=np.maximum(dual.x,0)
        if not np.isfinite(a).all() or a.sum()<=0:return None
        a/=a.sum();bound=gap(a)
        if not np.isfinite(bound) or bound < -tol:return None
        return dict(objective_gap_bound=float(max(0,bound)),objective_lower_bound=value(w)-float(max(0,bound)),
                    group_multipliers=a.tolist(),dual_termination=str(dual.message))
    attempts=[];starts=[u]
    for v in initial_weights:
        if np.shape(v)==(h,) and np.isfinite(v).all():starts.insert(0,simplex_projection(v))
    best=None
    def inspect(w,label):
        nonlocal best
        if not np.isfinite(w).all() or w.min() < -tol or abs(w.sum()-1)>tol:return None
        w=np.maximum(w,0);w/=w.sum();cert=certificate(w)
        record=dict(method=label,objective=value(w),certificate=cert);attempts.append(record)
        if best is None or record['objective']<best[0]:best=(record['objective'],w.copy())
        if cert is not None and cert['objective_gap_bound']<=tol and value(w)<=value(u)+tol:
            return w,dict(success=True,method='recovery-certified-SLSQP',objective=value(w),max_group_nll=float(losses(w).max()),
                group_nll=dict(zip(['00','01','10','11'],losses(w).tolist())),certificate=cert,
                feasibility_tolerance=tol,attempts=attempts,training_data_only=True,
                note='Same objective/floor/penalty/simplex and original acceptance tolerance; independent dual certificate. No test inputs used.')
        return None
    for i,w in enumerate(starts):
        accepted=inspect(w,f'saved-solver-start{i}')
        if accepted:return accepted
    for i in range(4):
        w=best[1] if best is not None else u
        # Start at a feasible epigraph point; use progressively tighter precision.
        z=np.r_[w,losses(w).max()+1e-10]
        result=minimize(lambda z:z[-1]+penalty*np.square(z[:-1]-u).sum(),z,
            jac=lambda z:np.r_[2*penalty*(z[:-1]-u),1.],method='SLSQP',
            bounds=[(0.,1.)]*h+[(0.,None)],constraints=[
                dict(type='eq',fun=lambda z:z[:-1].sum()-1,jac=lambda z:np.r_[np.ones(h),0.]),
                dict(type='ineq',fun=lambda z:z[-1]-losses(z[:-1]),jac=lambda z:np.c_[-gradients(z[:-1]),np.ones(4)])],
            options=dict(maxiter=max(4000,solver['maxiter']),ftol=[1e-12,1e-13,1e-14,1e-14][i]))
        accepted=inspect(result.x[:-1],f'polished-SLSQP{i}: {result.message}')
        if accepted:return accepted
    raise ValueError('Recovery retained original1e-7 acceptance tolerance but could not certify this fit: '+json.dumps(attempts))


def partition_preflight(work,worker,settings,rules):
    """Check every available partition's joint solves before resuming any CNN."""
    import ast,re
    from src.methods.dpe_head_reweighting import ensemble
    tag=settings['MODE']+'-'+hashlib.sha256((json.dumps(settings,sort_keys=True)+hashlib.sha256(worker.read_bytes()).hexdigest()).encode()).hexdigest()[:10]
    out=work/'results'/tag
    plans=list((work/'original_evidence').glob('dpe_head_aggregation_session/studies/*/plan.json'))
    if len(plans)!=1:raise RuntimeError('Expected the restored original study plan before recovery preflight')
    config=json.loads(plans[0].read_text())['config']
    ns=dict(np=np,GROUPS=['00','01','10','11'])
    node=next(n for n in ast.parse(worker.read_text()).body if isinstance(n,ast.FunctionDef) and n.name=='partition_rows')
    exec(compile(ast.Module(body=[node],type_ignores=[]),str(worker),'exec'),ns)
    checked=[]
    for path in sorted((out/'control_heads').glob('seed*-partition*.pt')):
        match=re.fullmatch(r'seed(\d+)-partition(\d+)\.pt',path.name)
        if not match:continue
        seed,split=map(int,match.groups());marker=path.with_suffix('.identity.json')
        lock=json.loads(marker.read_text())
        if hashlib.sha256(path.read_bytes()).hexdigest()!=lock['sha256']:raise ValueError('Saved partition head hash mismatch')
        # Completed frozen fits already passed the original checks; do not refit them.
        if (out/'frozen'/f'partition{split}-seed{seed}.json').is_file():continue
        source_root=work/'original_evidence/dpe_head_aggregation_session' if seed<3 else work/'additional_sources'
        manifests=list((source_root/'head_sources').glob(f'standard-seed{seed}-*/experiment/manifest.json'))
        if len(manifests)!=1:raise ValueError('Ambiguous source for partition preflight')
        manifest=json.loads(manifests[0].read_text());rows=ns['partition_rows'](manifest['validation'],split)['combiner_training']
        features=list((source_root/'features/standard').glob(f'seed{seed}-*/val_raw.npz'))
        if len(features)!=1:raise ValueError('Ambiguous validation features for partition preflight')
        # Validate the restored raw validation identity before feature-only prediction.
        handoff=json.loads((features[0].parent/'handoff.json').read_text())
        if hashlib.sha256(features[0].read_bytes()).hexdigest()!=handoff['artifacts']['val_raw.npz']:raise ValueError('Validation feature hash mismatch')
        with np.load(features[0],allow_pickle=False) as f:raw=f['features'];ids=f['ids'].astype(str).tolist()
        if ids!=[r['id'] for r in manifest['validation']]:raise ValueError('Validation feature order mismatch')
        lookup={name:i for i,name in enumerate(ids)};ix=[lookup[r['id']] for r in rows]
        # Match the actual worker's GPU computation, without images or CNN training.
        head=ensemble.load(path,2048,15);p=ensemble.predict(head,raw[ix],device='cuda')
        y=np.array([r['y'] for r in rows]);g=np.array([r['group'] for r in rows])
        for penalty in config['weighting_rules']['joint_lambdas']:
            print(f'PREFLIGHT saved partition seed{seed}/split{split}, joint lambda={penalty}',flush=True)
            w,fit=rules.joint_weights(p,y,g,penalty,config['weighting_rules']['solver'])
            checked.append(dict(source_seed=seed,partition_seed=split,penalty=penalty,fit=fit))
    (work/'logs/solver_partition_preflight.json').write_text(json.dumps(dict(status='passed',checked=checked),indent=2)+'\n')
    print('Saved partition solver preflight PASSED; joint fits checked:',len(checked),flush=True)


def launch(worker,config):
    worker=Path(worker).resolve();config=Path(config).resolve();work=config.parent
    settings=json.loads(config.read_text());repo=work/'reference_repo'
    os.chdir(repo);sys.path.insert(0,str(repo))
    from src.methods.dpe_head_reweighting import weighting_rules as rules
    if settings.get('MODE')=='five_seed_verification':
        import platform,subprocess,torch,torchvision,scipy,PIL
        tag=settings['MODE']+'-'+hashlib.sha256((json.dumps(settings,sort_keys=True)+hashlib.sha256(worker.read_bytes()).hexdigest()).encode()).hexdigest()[:10]
        environment=work/'results'/tag/'environment.json'
        current=dict(python=platform.python_version(),torch=str(torch.__version__),torchvision=str(torchvision.__version__),numpy=np.__version__,scipy=scipy.__version__,
            pillow=PIL.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            commit=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(),runner_sha256=hashlib.sha256(worker.read_bytes()).hexdigest())
        prior=json.loads(environment.read_text())
        mismatch={k:dict(saved=prior.get(k),current=v) for k,v in current.items() if prior.get(k)!=v}
        if mismatch:raise RuntimeError('Runtime identity differs. Stopping BEFORE changing old reports or starting training: '+json.dumps(mismatch))
        torch.set_num_threads(settings['CPU_THREADS'])
    original=rules.joint_weights;original_minimize=rules.minimize
    audit=work/'logs/solver_recovery.jsonl';audit.parent.mkdir(exist_ok=True)
    def recovered(p,y,groups,penalty,solver):
        captured=[]
        def tracked(*args,**kwargs):
            result=original_minimize(*args,**kwargs);captured.append(result);return result
        rules.minimize=tracked
        try:return original(p,y,groups,penalty,solver)
        except ValueError as exc:
            if not str(exc).startswith('Joint solver failed; no candidate accepted'):raise
            original_error=str(exc)
        finally:rules.minimize=original_minimize
        # Only a rejected fit is retried. Retain cached successful fits verbatim.
        p=rules.probabilities(p)
        for head in p:rules.metrics(head,y,groups)
        key=hashlib.sha256(np.ascontiguousarray(p).tobytes()+np.asarray(y).tobytes()+np.asarray(groups).astype(str).tobytes()+str(penalty).encode()).hexdigest()[:20]
        inputs=work/'logs'/('rejected-joint-'+key+'.npz')
        if not inputs.exists():np.savez_compressed(inputs,probabilities=p,labels=y,groups=np.asarray(groups).astype(str),penalty=penalty,solver_json=json.dumps(solver))
        print('RECOVERY: independently certifying the SAME joint objective, lambda='+str(penalty),flush=True)
        try:
            w,fit=certified_refit(p,y,groups,penalty,solver,[r.x[:-1] for r in captured if hasattr(r,'x')])
        except Exception as exc:
            with audit.open('a') as f:f.write(json.dumps(dict(time=time.time(),penalty=penalty,original_error=original_error,status='failed',error=str(exc)))+'\n')
            raise
        fit['rejected_training_inputs']=inputs.name
        fit['original_error']=original_error;fit['recovery_adapter_sha256']=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()
        with audit.open('a') as f:f.write(json.dumps(dict(time=time.time(),penalty=penalty,status='certified',fit=fit))+'\n')
        print('RECOVERY certified objective-gap bound:',fit['certificate']['objective_gap_bound'],flush=True)
        return w,fit
    rules.joint_weights=recovered
    provenance=dict(worker_sha256=hashlib.sha256(worker.read_bytes()).hexdigest(),config_sha256=hashlib.sha256(config.read_bytes()).hexdigest(),
        adapter_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
        cached_results_policy='Keep original worker/config bytes, result context and successful fits; record solver-only fallback for rejected fits.',
        unchanged='Backbones/heads/data/model/penalty/probability floor/objective/acceptance tolerance',
        changed='Retry failed joint optimizer and independently certify its global objective gap; never use test performance')
    (work/'logs/solver_recovery_provenance.json').write_text(json.dumps(provenance,indent=2)+'\n')
    # Match the original worker's __file__, argv, RUN_TAG and environment fingerprint.
    # Cache identity therefore remains valid; separate recovery provenance records the adapter.
    if settings.get('MODE')=='five_seed_verification':partition_preflight(work,worker,settings,rules)
    sys.argv=[str(worker),str(config)];runpy.run_path(str(worker),run_name='__main__')


if __name__=='__main__':launch(sys.argv[1],sys.argv[2])
'''
recovery_path=work/'solver_recovery.py'
if recovery_path.exists() and recovery_path.read_text()!=RECOVERY_SOURCE:
    saved=recovery_path.with_name('solver_recovery.previous-'+hashlib.sha256(recovery_path.read_bytes()).hexdigest()[:12]+'.py')
    if not saved.exists():shutil.copy2(recovery_path,saved)
recovery_path.write_text(RECOVERY_SOURCE)
print('Recovery adapter:',recovery_path)


## Resume

Before any CNN resumes, the adapter checks the exact saved runtime identity and retries joint fits for every saved partition whose fit cache is missing, including the failed partition. This uses saved features and heads on Kaggle. If this preflight fails, no backbone training starts.

The original 8.5-hour worker budget restarts for this session. This does not imply another 8.5 hours is needed: time depends on how many backbone seeds were already completed. Logs should say `Reusing completed shared ERM` for each finished new backbone. Cheap control fits may repeat where the error prevented an atomic cache file from being written.

If the independent certificate still cannot pass, the run stops and preserves evidence; do not relax the tolerance or accept a result based on its test score. Run the final cell manually after a failure to download partial reports. Keep the full output in all cases.


In [ ]:
import signal

logs = work / "logs"
logs.mkdir(exist_ok=True)
env = os.environ.copy()
env["PYTHONPATH"] = str(runtime) + os.pathsep + str(repo) + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
env["OMP_NUM_THREADS"] = str(C["CPU_THREADS"])
env["MKL_NUM_THREADS"] = str(C["CPU_THREADS"])
env["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"
log_path = logs / ("solver_recovery_" + str(__import__("time").time_ns()) + ".log")
with log_path.open("w", buffering=1) as logfile:
    process = subprocess.Popen([sys.executable, "-u", str(recovery_path), str(runner_path), str(config_path)],
                               cwd=repo, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            logfile.write(line)
        returncode = process.wait()
    except KeyboardInterrupt:
        process.send_signal(signal.SIGINT)
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.terminate()
            process.wait()
        raise
if returncode:
    raise RuntimeError(f"Verification stopped with exit code {returncode}. Inspect {log_path}. Run the download cell below to preserve any partial results.")
print("Finished. Download the results below.")


In [ ]:
from IPython.display import FileLink, Markdown, display

result_zip = work / "verification_results.zip"
if result_zip.is_file():
    display(FileLink(str(result_zip), result_html_prefix="Download compact verification results: "))
else:
    print("No archive exists yet. Check the setup/worker log and preserve the working directory.")
for report in sorted((work / "results").glob("*/report.md")):
    display(Markdown(f"### {report.parent.name}"))
    display(Markdown(report.read_text()))
print("For backbone resume, preserve the full notebook output, not only the compact ZIP.")
